# sensor-pulse — Baseline RUL Prediction
**Dataset:** NASA CMAPSS Turbofan Engine Degradation (FD001–FD004)  
**Stack:** PySpark · Scikit-learn · Random Forest · Matplotlib

Predicts Remaining Useful Life (RUL) for turbofan engines using a Random Forest
baseline model trained on Gold-layer features built from the Silver pipeline.

| Section | What it does |
|---------|-------------|
| 1–4 | Setup, data download, Silver rebuild, Gold features |
| 5 | FD001 baseline — engine-based split, RF, RMSE/MAE, plots |
| 6 | FD002–FD004 — per-operating-condition normalization |
| 7 | Summary and next steps |

> **Data leakage note:** `max_cycle` (used to compute RUL) is derived from the
> full engine run-to-failure trajectory. In production this would be unknown —
> the engine hasn't failed yet. This is the standard CMAPSS training assumption
> (right-censored data problem). Any model trained here implicitly knows the
> failure horizon, so reported metrics are optimistic relative to a live system.

## 1. Setup

In [ ]:
!pip install pyspark kagglehub matplotlib scikit-learn -q

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from pyspark.sql.window import Window
import kagglehub
import shutil
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.preprocessing import StandardScaler

spark = SparkSession.builder \
    .appName("SensorPulse_RUL") \
    .config("spark.sql.shuffle.partitions", "4") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")
print(f"Spark version: {spark.version}")

## 2. Download Data (All 4 Datasets)

In [ ]:
path = kagglehub.dataset_download("behrad3d/nasa-cmaps")
os.makedirs("data/raw", exist_ok=True)

for ds in ["FD001", "FD002", "FD003", "FD004"]:
    shutil.copy(f"{path}/CMaps/train_{ds}.txt", f"data/raw/train_{ds}.txt")
    print(f"  data/raw/train_{ds}.txt ready")

print("\nAll 4 datasets ready")

## 3. Schema

In [ ]:
sensor_cols = [StructField(f"sensor_{i:02d}", DoubleType(), True) for i in range(1, 22)]

schema = StructType([
    StructField("engine_id",    IntegerType(), True),
    StructField("cycle",        IntegerType(), True),
    StructField("op_setting_1", DoubleType(),  True),
    StructField("op_setting_2", DoubleType(),  True),
    StructField("op_setting_3", DoubleType(),  True),
] + sensor_cols)

print(f"Schema: {len(schema.fields)} columns")

## 4. Rebuild Silver from Raw CSVs

Same validation pattern as the main pipeline: null checks on key sensors,
status flags kept (rows never deleted), `max_cycle` and `rul` derived per engine.

In [ ]:
KEY_SENSORS = ["sensor_02", "sensor_03", "sensor_04", "sensor_07", "sensor_11"]

def build_silver(dataset_id):
    """Read raw CMAPSS file and apply Silver validation rules."""
    df_raw = spark.read \
        .schema(schema) \
        .option("sep", " ") \
        .option("header", "false") \
        .csv(f"data/raw/train_{dataset_id}.txt")

    null_cond = F.lit(False)
    for col in ["engine_id", "cycle"] + KEY_SENSORS:
        null_cond = null_cond | F.col(col).isNull()

    window_engine = Window.partitionBy("engine_id")

    df_silver = df_raw \
        .withColumn(
            "status",
            F.when(null_cond, F.lit("invalid_null")).otherwise(F.lit("valid"))
        ) \
        .withColumn("max_cycle", F.max("cycle").over(window_engine)) \
        .withColumn("cycle_pct",  F.round(F.col("cycle") / F.col("max_cycle"), 3)) \
        .withColumn("rul",        (F.col("max_cycle") - F.col("cycle")).cast("double"))

    valid_count = df_silver.filter(F.col("status") == "valid").count()
    engines     = df_silver.select("engine_id").distinct().count()
    print(f"  {dataset_id}: {valid_count:,} valid rows | {engines} engines")
    return df_silver


print("Building Silver for all datasets...")
silver = {ds: build_silver(ds) for ds in ["FD001", "FD002", "FD003", "FD004"]}

## 5. Build Gold ML Features

Features match the Gold layer in `sensor_pulse_pipeline.ipynb` and the
column definitions in `stg_sensor_readings.sql`:

| Feature | Description |
|---------|-------------|
| `sensor_02/04/11` | Raw sensor readings (fan speed, HPC pressure, fan efficiency proxy) |
| `s02_mean_10` | 10-cycle rolling mean of sensor_02 — smoothed trend |
| `s02_std_10` | 10-cycle rolling std of sensor_02 — signal volatility |
| `s02_delta` | Cycle-over-cycle change in sensor_02 — rate of change |
| `s04_delta` | Cycle-over-cycle change in sensor_04 |
| `cycle_pct` | Position in engine life (0 = start, 1 = failure) |

In [ ]:
def build_gold_features(df_silver, dataset_id):
    """Compute ML features via Spark SQL window functions."""
    df_valid = df_silver.filter(F.col("status") == "valid")
    view = f"silver_{dataset_id.lower()}"
    df_valid.createOrReplaceTempView(view)

    gold = spark.sql(f"""
        SELECT
            engine_id,
            cycle,
            cycle_pct,
            op_setting_1,
            op_setting_2,
            sensor_02,
            sensor_04,
            sensor_11,
            ROUND(AVG(sensor_02) OVER (
                PARTITION BY engine_id ORDER BY cycle
                ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
            ), 4) AS s02_mean_10,
            ROUND(STDDEV(sensor_02) OVER (
                PARTITION BY engine_id ORDER BY cycle
                ROWS BETWEEN 9 PRECEDING AND CURRENT ROW
            ), 5) AS s02_std_10,
            ROUND(sensor_02 - LAG(sensor_02, 1) OVER (
                PARTITION BY engine_id ORDER BY cycle
            ), 5) AS s02_delta,
            ROUND(sensor_04 - LAG(sensor_04, 1) OVER (
                PARTITION BY engine_id ORDER BY cycle
            ), 5) AS s04_delta,
            rul
        FROM {view}
        ORDER BY engine_id, cycle
    """)

    print(f"  {dataset_id}: {gold.count():,} feature rows")
    return gold


print("Building Gold ML features...")
gold = {ds: build_gold_features(silver[ds], ds) for ds in ["FD001", "FD002", "FD003", "FD004"]}

## 6. FD001 Baseline — Random Forest RUL Model

FD001 is the simplest CMAPSS dataset: single operating condition, single fault mode.
It gives a clean baseline before handling the complexity in FD002–FD004.

### 6a. Engine-Based Train/Test Split (No Data Leakage)

**Why engine-based?** Splitting randomly on rows would put early and late cycles
of the *same engine* into both train and test sets. The model would then memorise
engine-specific degradation patterns rather than generalise. Splitting on engines
ensures the test set contains engines the model has never seen.

FD001 has 100 engines → engines 1–80 train, 81–100 test.

In [ ]:
FEATURE_COLS = [
    "sensor_02", "sensor_04", "sensor_11",
    "s02_mean_10", "s02_std_10", "s02_delta", "s04_delta",
    "cycle_pct"
]
RUL_CAP = 125  # Standard CMAPSS cap — ignore very early life where RUL is noisy

df_fd001 = gold["FD001"].toPandas()

# Cap RUL: early-life cycles contribute little signal, capping reduces noise
df_fd001["rul"] = df_fd001["rul"].clip(upper=RUL_CAP)

TRAIN_ENGINES = list(range(1, 81))
TEST_ENGINES  = list(range(81, 101))

df_train = df_fd001[df_fd001["engine_id"].isin(TRAIN_ENGINES)].dropna(subset=FEATURE_COLS).copy()
df_test  = df_fd001[df_fd001["engine_id"].isin(TEST_ENGINES)].dropna(subset=FEATURE_COLS).copy()

print(f"Train: {len(df_train):,} rows | {df_train['engine_id'].nunique()} engines")
print(f"Test:  {len(df_test):,} rows  | {df_test['engine_id'].nunique()} engines")
print(f"RUL distribution (train): min={df_train['rul'].min():.0f}, "
      f"max={df_train['rul'].max():.0f}, mean={df_train['rul'].mean():.1f}")

### 6b. Normalise and Train Random Forest

In [ ]:
X_train = df_train[FEATURE_COLS].values
y_train = df_train["rul"].values
X_test  = df_test[FEATURE_COLS].values
y_test  = df_test["rul"].values

# Fit scaler on training data only — prevent test leakage
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

rf = RandomForestRegressor(
    n_estimators=100,
    max_depth=12,
    min_samples_leaf=5,
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_sc, y_train)
print("Random Forest trained")

### 6c. Evaluate — RMSE and MAE

In [ ]:
y_pred = rf.predict(X_test_sc)

rmse = np.sqrt(mean_squared_error(y_test, y_pred))
mae  = mean_absolute_error(y_test, y_pred)

print(f"FD001 Baseline — Random Forest")
print(f"  RMSE : {rmse:.2f} cycles")
print(f"  MAE  : {mae:.2f} cycles")
print()
print("Reference: SOTA models on FD001 typically achieve RMSE ~12-16 cycles.")
print("A baseline RF in the 20-30 range is a solid starting point.")

### 6d. Predicted vs Actual RUL — Scatter Plot

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))

ax.scatter(y_test, y_pred, alpha=0.25, s=8, color="steelblue", label="Prediction")
ax.plot([0, RUL_CAP], [0, RUL_CAP], "r--", lw=1.5, label="Perfect prediction")

ax.set_xlabel("Actual RUL (cycles)")
ax.set_ylabel("Predicted RUL (cycles)")
ax.set_title(f"FD001 — Predicted vs Actual RUL\nRMSE={rmse:.1f}  MAE={mae:.1f}")
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig("rul_scatter_fd001.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rul_scatter_fd001.png")

### 6e. Feature Importance

In [ ]:
importances = pd.Series(rf.feature_importances_, index=FEATURE_COLS).sort_values()

fig, ax = plt.subplots(figsize=(7, 4))
importances.plot.barh(ax=ax, color="steelblue", edgecolor="white")
ax.set_xlabel("Mean Decrease in Impurity")
ax.set_title("FD001 — Random Forest Feature Importance")
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.savefig("feature_importance_fd001.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: feature_importance_fd001.png")
print()
print("Top features:")
print(importances[::-1].to_string())

## 7. Generalise to FD002–FD004

FD002 and FD004 have **six discrete operating conditions** (combinations of
throttle resolver angle and altitude). Raw sensor values shift between conditions,
so a global scaler would embed operating-condition bias into the features.

**Fix:** fit a StandardScaler per operating condition cluster (identified by rounding
`op_setting_1` to 2 decimal places — CMAPSS uses exactly 6 discrete settings).

FD001 and FD003 are single-condition → the per-condition approach degrades
gracefully to a global scaler.

In [ ]:
def get_op_cond(df):
    """Assign a discrete operating-condition label by rounding op settings."""
    return (
        df["op_setting_1"].round(2).astype(str) + "_" +
        df["op_setting_2"].round(2).astype(str)
    )


def fit_scalers_per_cond(df_train, feature_cols):
    """Fit one StandardScaler per operating condition on training data."""
    scalers = {}
    for cond in df_train["op_cond"].unique():
        mask = df_train["op_cond"] == cond
        sc = StandardScaler()
        sc.fit(df_train.loc[mask, feature_cols])
        scalers[cond] = sc
    return scalers


def apply_scalers(df, scalers, feature_cols):
    """Apply pre-fitted per-condition scalers; fall back to global mean/std if unseen."""
    df = df.copy()
    # Aggregate fallback for any test condition not seen in training
    all_means = np.mean([sc.mean_ for sc in scalers.values()], axis=0)
    all_stds  = np.mean([sc.scale_ for sc in scalers.values()], axis=0)

    X = np.zeros((len(df), len(feature_cols)))
    for i, (_, row_group) in enumerate(df.groupby(df.index, sort=False)):
        # group is single row here; iterate via condition
        pass  # replaced below with vectorised apply

    # Vectorised: apply each condition's scaler
    result = np.zeros((len(df), len(feature_cols)))
    for cond, sc in scalers.items():
        mask = (df["op_cond"] == cond).values
        if mask.any():
            result[mask] = sc.transform(df.loc[mask, feature_cols].values)
    # Fill rows with unseen conditions using global fallback
    seen = df["op_cond"].isin(scalers).values
    if (~seen).any():
        raw = df.loc[~seen, feature_cols].values
        result[~seen] = (raw - all_means) / (all_stds + 1e-8)

    return result


print("Per-condition normalisation helpers defined")

In [ ]:
results = {}

for ds in ["FD001", "FD002", "FD003", "FD004"]:
    print(f"\n{'='*40}")
    print(f"Dataset: {ds}")

    df = gold[ds].toPandas()
    df["rul"] = df["rul"].clip(upper=RUL_CAP)
    df["op_cond"] = get_op_cond(df)

    engine_ids = sorted(df["engine_id"].unique())
    n_engines  = len(engine_ids)
    split_idx  = int(n_engines * 0.8)
    train_engines = engine_ids[:split_idx]
    test_engines  = engine_ids[split_idx:]

    df_train = df[df["engine_id"].isin(train_engines)].dropna(subset=FEATURE_COLS).copy()
    df_test  = df[df["engine_id"].isin(test_engines)].dropna(subset=FEATURE_COLS).copy()

    print(f"  Engines total: {n_engines}  |  train: {len(train_engines)}, test: {len(test_engines)}")
    print(f"  Operating conditions in train: {df_train['op_cond'].nunique()}")

    # Per-condition normalisation
    scalers = fit_scalers_per_cond(df_train, FEATURE_COLS)
    X_train = apply_scalers(df_train, scalers, FEATURE_COLS)
    X_test  = apply_scalers(df_test,  scalers, FEATURE_COLS)
    y_train = df_train["rul"].values
    y_test  = df_test["rul"].values

    model = RandomForestRegressor(
        n_estimators=100,
        max_depth=12,
        min_samples_leaf=5,
        random_state=42,
        n_jobs=-1
    )
    model.fit(X_train, y_train)

    y_pred = model.predict(X_test)
    rmse   = np.sqrt(mean_squared_error(y_test, y_pred))
    mae    = mean_absolute_error(y_test, y_pred)

    results[ds] = {"rmse": rmse, "mae": mae, "model": model}
    print(f"  RMSE: {rmse:.2f}  |  MAE: {mae:.2f}")

In [ ]:
print("\n" + "="*45)
print(f"{'Dataset':<10} {'RMSE':>8} {'MAE':>8}")
print("-"*45)
for ds, r in results.items():
    print(f"{ds:<10} {r['rmse']:>8.2f} {r['mae']:>8.2f}")
print("="*45)
print()
print("FD001/FD003 = single operating condition (easiest)")
print("FD002/FD004 = 6 operating conditions (harder — more variance to explain)")

### Scatter plots — all four datasets

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, ds in zip(axes.flat, ["FD001", "FD002", "FD003", "FD004"]):
    df = gold[ds].toPandas()
    df["rul"]     = df["rul"].clip(upper=RUL_CAP)
    df["op_cond"] = get_op_cond(df)

    engine_ids   = sorted(df["engine_id"].unique())
    split_idx    = int(len(engine_ids) * 0.8)
    test_engines = engine_ids[split_idx:]

    df_test = df[df["engine_id"].isin(test_engines)].dropna(subset=FEATURE_COLS).copy()

    # Re-build scalers for this dataset (reuse from loop above via stored model)
    train_engines = engine_ids[:split_idx]
    df_train = df[df["engine_id"].isin(train_engines)].dropna(subset=FEATURE_COLS).copy()
    scalers  = fit_scalers_per_cond(df_train, FEATURE_COLS)
    X_test   = apply_scalers(df_test, scalers, FEATURE_COLS)
    y_test   = df_test["rul"].values
    y_pred   = results[ds]["model"].predict(X_test)

    r = results[ds]
    ax.scatter(y_test, y_pred, alpha=0.2, s=6, color="steelblue")
    ax.plot([0, RUL_CAP], [0, RUL_CAP], "r--", lw=1.5)
    ax.set_title(f"{ds}   RMSE={r['rmse']:.1f}  MAE={r['mae']:.1f}")
    ax.set_xlabel("Actual RUL")
    ax.set_ylabel("Predicted RUL")
    ax.grid(True, alpha=0.3)

plt.suptitle("Predicted vs Actual RUL — Random Forest Baseline", fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig("rul_scatter_all.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: rul_scatter_all.png")

## Summary

| What | Detail |
|------|--------|
| Model | Random Forest (100 trees, max_depth=12) |
| Features | 8 engineered features from Gold layer |
| RUL cap | 125 cycles — standard CMAPSS baseline |
| Split | Engine-based (80/20) — no data leakage |
| Normalisation | Per operating condition for FD002/FD004 |

**Known limitations:**
1. `max_cycle` (failure time) is only known because we have complete run-to-failure
   trajectories — a deployed system would not know this. This makes reported metrics
   optimistic vs a real-time prediction system.
2. Random Forest does not model temporal dependencies — it treats each cycle
   as an independent observation. An LSTM or Transformer would capture sequence
   patterns better.
3. Feature engineering (delta, rolling stats) partially compensates for (2).

**Next steps:**
- Add XGBoost and compare RMSE
- Try LSTM on raw sensor sequence (no feature engineering needed)
- Apply NASA's PHMAP scoring function (penalises late predictions more than early)
- Export model artifacts to `data/models/` for serving